In [1]:
import pandas as pd
df=pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
#tokenization
def tokenize(text):
  text=text.lower()
  text=text.replace('?','')
  text=text.replace("'","")
  return text.split()


In [3]:
#vocabulary
vocab={"<UNK>":0}

In [4]:
def build_vocab(row):
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  merged_token=tokenized_question+tokenized_answer
  for token in merged_token:
    if(token not in vocab):
      vocab[token]=len(vocab)

In [5]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [6]:
len(vocab)

324

In [7]:
#convert words to numerical indices
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab["<UNK>"])
  return indexed_text

In [8]:
import torch
from torch.utils.data import Dataset,DataLoader

In [9]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,idx):
    numerical_question=text_to_indices(self.df.iloc[idx]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[idx]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [10]:
dataset=QADataset(df,vocab)

In [11]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [12]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [13]:
import torch.nn as nn
class simpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,question):
    embedded_question=self.embedding(question)
    hidden,final=self.rnn(embedded_question)
    output=self.fc(final.squeeze(0))
    return output



In [14]:
x=nn.Embedding(324,embedding_dim=50)
y=nn.RNN(50,64,batch_first=True)
z=nn.Linear(64,324)

a=dataset[0][0].reshape(1,6)
print("shape of a: ",a.shape)
b=x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)


shape of a:  torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])


In [15]:
e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of e: torch.Size([1, 324])


In [16]:
learning_rate=0.001
epochs=20

In [17]:
model=simpleRNN(len(vocab))

In [18]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [20]:
#training loop
for epoch in range (epochs):
  total_loss=0
  for question,answer in dataloader:
    optimizer.zero_grad()
    #forward pass
    output=model(question)
    #loss
    loss=loss_fn(output,answer[0])
    #backward
    loss.backward()
    #update
    optimizer.step()

    total_loss=total_loss+loss.item()
  print(f"Epoch: {epoch+1} , Loss: {total_loss:4f}")

Epoch: 1 , Loss: 524.920317
Epoch: 2 , Loss: 458.453073
Epoch: 3 , Loss: 381.453321
Epoch: 4 , Loss: 319.475550
Epoch: 5 , Loss: 267.915754
Epoch: 6 , Loss: 219.693838
Epoch: 7 , Loss: 176.035683
Epoch: 8 , Loss: 138.396072
Epoch: 9 , Loss: 106.661829
Epoch: 10 , Loss: 82.543047
Epoch: 11 , Loss: 64.568195
Epoch: 12 , Loss: 51.111385
Epoch: 13 , Loss: 40.922073
Epoch: 14 , Loss: 33.325641
Epoch: 15 , Loss: 27.841905
Epoch: 16 , Loss: 23.334327
Epoch: 17 , Loss: 19.822487
Epoch: 18 , Loss: 16.970406
Epoch: 19 , Loss: 14.707333
Epoch: 20 , Loss: 12.534681


In [33]:
def predict(model,question,threshold=0.5):
  numerical_question=text_to_indices(question,vocab)
  question_tensor=torch.tensor(numerical_question).unsqueeze(0)
  output=model(question_tensor)
  #convert logits to probabilities
  probs=torch.nn.functional.softmax(output,dim=1)
  #find index of max probability
  value,index=torch.max(probs,dim=1)
  print(value,index)

  if value<threshold:
    print("I don't know")
  else:
    # Corrected line: First get the list of keys, then index it with the integer value from the tensor
    print(list(vocab.keys())[index.item()])

In [32]:
predict(model,"what is capital of france")

tensor([0.8693], grad_fn=<MaxBackward0>) tensor([7])
paris
